# Notebook 7 · A simple language task: character-level next-token prediction

Companion to lectures [3](https://jiangyou2025.github.io/kun/course/03/) and [9](https://jiangyou2025.github.io/kun/course/09/).

Forecasting and language modelling are the **same problem**: given a sequence, predict what comes
next. In the earlier notebooks the sequence was a number per time step; here each step is a
**character**, and "predict the next value" becomes "predict the next character". Everything stays
**self-contained** (a tiny built-in text, nothing to download):

1. turn text into integers — build a **vocabulary** and encode/decode;
2. a **bigram** baseline from pure counts (`numpy`), scored by average negative log-likelihood;
3. the **same bigram as a neural net** (`torch`): one weight matrix trained with cross-entropy —
   it should rediscover the count probabilities;
4. give the model **context** (the last few characters) with a small MLP, and watch the loss drop;
5. **generate** new text by sampling one character at a time.

> Requires: `numpy`, `matplotlib`, `torch`

## 0. A tiny built-in corpus

No dataset to download — we hand-write a small, repetitive text. Repetition gives the model clear
structure to latch onto, so even a tiny model produces recognisable patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A small, self-contained corpus (themed on the course, repeated for structure).
text = (
    "time series forecasting predicts the next value from the past. "
    "a language model predicts the next character from the past. "
    "the past predicts the future. the future follows the past. "
    "trend and season repeat; patterns repeat; the model learns to repeat. "
) * 12

print("characters:", len(text))
print("preview   :", text[:80], "...")

## 1. Vocabulary: text -> integers

A model eats numbers, not letters. We list every distinct character (the **vocabulary**) and map
each to an index. `encode` turns a string into a list of indices; `decode` turns it back.

In [ ]:
chars = sorted(set(text))
V = len(chars)                       # vocabulary size
stoi = {c: i for i, c in enumerate(chars)}   # char  -> index
itos = {i: c for c, i in stoi.items()}       # index -> char

encode = lambda s: [stoi[c] for c in s]
decode = lambda ix: "".join(itos[i] for i in ix)

print("vocab size:", V)
print("vocab     :", "".join(chars).replace("\n", "\\n"))
print("encode('time') ->", encode("time"), "-> decode ->", decode(encode("time")))

## 2. Bigram baseline from counts (numpy)

The simplest language model: the next character depends **only on the current one**. Count every
pair `(a, b)` that appears, then normalise each row to a probability. This is the language analogue
of the AR/linear baseline in [Notebook 2](02_linear_from_scratch.ipynb) — no training loop, just a
closed-form count.

We **score** a model by its average **negative log-likelihood (NLL)** per character — the
cross-entropy loss. Lower is better; `log(V)` is the score of pure guessing.

In [ ]:
# Count every adjacent pair (a -> b)
N = np.zeros((V, V), dtype=np.float64)
for a, b in zip(text, text[1:]):
    N[stoi[a], stoi[b]] += 1

# Add-1 (Laplace) smoothing so unseen pairs are not impossible, then normalise rows.
P = (N + 1.0)
P /= P.sum(axis=1, keepdims=True)

def nll(probs_of_next):
    """Average negative log-likelihood per character under a bigram table."""
    ll, n = 0.0, 0
    for a, b in zip(text, text[1:]):
        ll += np.log(probs_of_next[stoi[a], stoi[b]])
        n += 1
    return -ll / n

print(f"bigram NLL      : {nll(P):.4f}")
print(f"random guessing : {np.log(V):.4f}")

Visualise the transition table: bright cells are likely `current -> next` pairs (e.g. `q` rarely
appears here, but a space is very likely after `t`-words).

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(P, cmap="viridis")
plt.colorbar(label="P(next | current)")
plt.xticks(range(V), chars, fontsize=7)
plt.yticks(range(V), chars, fontsize=7)
plt.xlabel("next char"); plt.ylabel("current char")
plt.title("Bigram transition probabilities")
plt.tight_layout(); plt.show()

**Generate** by sampling: start from a character, draw the next one from its row of `P`, then
repeat. The bigram has no memory beyond one character, so the text is locally plausible but globally
nonsense.

In [ ]:
rng = np.random.default_rng(0)

def sample_bigram(n=200, start="t"):
    out = [start]
    i = stoi[start]
    for _ in range(n):
        i = rng.choice(V, p=P[i])
        out.append(itos[i])
    return "".join(out)

print(sample_bigram(200))

## 3. The same bigram as a neural net (torch)

Now train the **identical model** by gradient descent instead of counting. One weight matrix
`W` (`V x V`) maps the current character (one-hot) to **logits** over the next character;
`softmax` turns logits into probabilities; **cross-entropy** is the loss. Minimising it should
recover the count probabilities from Section 2 — same model, two routes (echoing the
normal-equation vs gradient-descent comparison in Notebook 2).

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

xs = torch.tensor(encode(text[:-1]))   # current char
ys = torch.tensor(encode(text[1:]))    # next char (target)

W = torch.randn(V, V, requires_grad=True)   # the only parameters

losses = []
for step in range(300):
    logits = W[xs]                     # (n, V): one row of W per input char
    loss = F.cross_entropy(logits, ys) # softmax + NLL in one call
    losses.append(loss.item())

    W.grad = None
    loss.backward()
    with torch.no_grad():
        W -= 30.0 * W.grad             # plain gradient descent

print(f"neural bigram NLL: {loss.item():.4f}   (count-based was {nll(P):.4f})")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(losses)
plt.axhline(nll(P), color="r", ls="--", label="count-based bigram NLL")
plt.xlabel("gradient step"); plt.ylabel("cross-entropy loss")
plt.title("Neural bigram converges to the count-based model")
plt.legend(); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## 4. Add context: a small MLP over the last `block` characters

A bigram only sees one character — that is why its samples drift. Give the model a **window** of
the last `block` characters (exactly the lag window `p` from the forecasting notebooks). We embed
each character, concatenate the window, and pass it through a one-hidden-layer **MLP**. More
context -> lower loss -> more coherent text.

In [ ]:
block = 4          # how many previous characters the model sees
n_emb = 8          # embedding size per character
n_hid = 64         # hidden units

# Build (context window -> next char) pairs.
data = encode(text)
Xc, Yc = [], []
for i in range(len(data) - block):
    Xc.append(data[i:i + block])
    Yc.append(data[i + block])
Xc = torch.tensor(Xc)          # (n, block)
Yc = torch.tensor(Yc)          # (n,)
print("contexts:", Xc.shape, " targets:", Yc.shape)

In [ ]:
torch.manual_seed(1)
C  = torch.randn(V, n_emb,            requires_grad=True)   # embedding table
W1 = torch.randn(block * n_emb, n_hid, requires_grad=True)
b1 = torch.zeros(n_hid,                requires_grad=True)
W2 = torch.randn(n_hid, V,             requires_grad=True)
b2 = torch.zeros(V,                    requires_grad=True)
params = [C, W1, b1, W2, b2]

def logits_of(Xb):
    emb = C[Xb].reshape(Xb.shape[0], -1)   # (n, block*n_emb)
    h   = torch.tanh(emb @ W1 + b1)        # (n, n_hid)
    return h @ W2 + b2                      # (n, V)

mlp_losses = []
for step in range(2000):
    logits = logits_of(Xc)
    loss = F.cross_entropy(logits, Yc)
    mlp_losses.append(loss.item())
    for p in params:
        p.grad = None
    loss.backward()
    with torch.no_grad():
        for p in params:
            p -= 1.0 * p.grad

print(f"MLP NLL: {mlp_losses[-1]:.4f}   (bigram was {nll(P):.4f})")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(mlp_losses, label="context MLP")
plt.axhline(nll(P), color="r", ls="--", label="bigram NLL")
plt.xlabel("gradient step"); plt.ylabel("cross-entropy loss")
plt.title(f"Context (block={block}) beats the memoryless bigram")
plt.legend(); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## 5. Generate from the context model

Same idea as before, but now we keep a sliding window of the last `block` characters and sample the
next one from the MLP's probabilities. With real context the output reproduces whole words and
phrases from the corpus instead of random letter soup.

In [ ]:
@torch.no_grad()
def sample_mlp(n=200, start="the past "):
    ctx = encode(start)[-block:]
    ctx = [0] * (block - len(ctx)) + ctx     # left-pad if too short
    out = list(start)
    for _ in range(n):
        logits = logits_of(torch.tensor([ctx]))
        probs = F.softmax(logits, dim=1).squeeze(0).numpy()
        i = rng.choice(V, p=probs)
        out.append(itos[i])
        ctx = ctx[1:] + [i]
    return "".join(out)

print(sample_mlp(220))

## Summary

- **Language modelling = forecasting on characters**: cut a sequence into `(context -> next)`
  pairs and predict the next token — the exact recipe from the time-series notebooks.
- A **bigram** is the language analogue of the AR(1) baseline; counting and gradient descent give
  the *same* model, scored by **cross-entropy / NLL** (lower than `log(V)` = better than guessing).
- **Context matters**: widening the window from 1 character (bigram) to `block` characters (MLP)
  lowers the loss and makes generated text coherent — the same reason a longer lag window helps a
  forecaster.
- This is the whole idea behind language models: predict the next token, then sample one at a time.
  Scaling it up — more context, deeper nets, attention — is how larger models work, just on more
  data and more tokens.